# Module 6 lab · Optimisation

The code from every lesson of [this module](https://aicourses.dev/mathematics/optimisation), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/mathematics/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 6.2 Gradient descent and the learning rate

One rule updates every knob, and one number, the learning rate, decides whether it works. On a bowl you can compute the step above which it flies away.

[Read the lesson](https://aicourses.dev/mathematics/optimisation/the-learning-rate)

### The digit classifier as a bowl

Lesson 05.9 gave the gradient of a layer for a batch, and squaring the miss adds a factor 2, so $\nabla L = \frac{2}{1297}\, \mathbf{A}^\top (\mathbf{A}\mathbf{P} - \mathbf{Y})$, where $\mathbf{A}$ holds the images with their 1s as rows and $\mathbf{Y}$ holds the targets. With every knob at 0 each score is 0, each image misses by 1 on its own class, so the loss is exactly $1^2 = 1$.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
np.seterr(over="ignore", invalid="ignore")   # runaway runs print inf
rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16
order = rng.permutation(len(X))
X, y = X[order], y[order]
Xtr, ytr, Xte, yte = X[:1297], y[:1297], X[1297:], y[1297:]
A = np.hstack([Xtr, np.ones((1297, 1))])     # images plus a column of ones
Y = np.eye(10)[ytr]                          # one-hot targets
loss = lambda P: ((A @ P - Y) ** 2).sum(axis=1).mean()
grad = lambda P: 2 / len(A) * A.T @ (A @ P - Y)
P = np.zeros((65, 10))                       # every knob at 0
print(A.shape, Y.shape, loss(P), np.linalg.norm(grad(P)).round(3))

*The lesson shows:*

```text
(1297, 65) (1297, 10) 1.0 2.308
```

### Three rates

Now the loop. It runs 500 steps and reports the loss after steps 10, 100 and 500, then the accuracy on the 500 held-out digits.

In [ ]:
Ate = np.hstack([Xte, np.ones((500, 1))])
acc = lambda P: ((Ate @ P).argmax(axis=1) == yte).mean()

def descend(lr, steps=500):
    P = np.zeros((65, 10))
    for i in range(1, steps + 1):
        P = P - lr * grad(P)
        if i in (10, 100, 500):
            print(f"  step {i}: loss {loss(P):.4g}")
    return P

for lr in [0.001, 0.08, 0.5]:
    print("rate", lr)
    P = descend(lr)
    print("  test accuracy", acc(P))

*The lesson shows:*

```text
rate 0.001
  step 10: loss 0.9555
  step 100: loss 0.8309
  step 500: loss 0.6497
  test accuracy 0.898
rate 0.08
  step 10: loss 0.5736
  step 100: loss 0.35
  step 500: loss 0.3147
  test accuracy 0.952
rate 0.5
  step 10: loss 2.387e+19
  step 100: loss 6.402e+202
  step 500: loss nan
  test accuracy 0.11
```

### The limit

Where is the edge between 0.08 and 0.5? Try rates one by one, 300 steps each:

In [ ]:
for lr in [0.080, 0.085, 0.087, 0.0875, 0.088, 0.090]:
    P = np.zeros((65, 10))
    for _ in range(300):
        P = P - lr * grad(P)
    print(lr, f"{loss(P):.4g}")

*The lesson shows:*

```text
0.08 0.3211
0.085 0.3201
0.087 0.3206
0.0875 1.108
0.088 712.7
0.09 2.252e+14
```

## 6.3 Stretched bowls: zig-zag and scaling

When a bowl is stretched the steepest direction sets the limit and the shallow one crawls. Rescaling the inputs rounds the bowl, so you see why preparing data matters.

[Read the lesson](https://aicourses.dev/mathematics/optimisation/stretched-bowls)

### A stretched bowl

The ratio of the steepest to the shallowest curvature, here $20 / 2 = 10$, is the bowl’s **condition number**. The bigger it is, the longer the crawl. The code counts the steps needed to bring the loss below 0.001 at the best rate, for the ratio 10 and for a ratio of 100 (the loss $x^2 + 100y^2$, curvatures 2 and 200):

In [ ]:
import numpy as np

def steps_needed(c, lr):
    x, y = 1.0, 1.0
    for i in range(1, 100000):
        x, y = x - lr * 2 * x, y - lr * c * y   # slopes 2x and c*y
        if x * x + c / 2 * y * y < 0.001:
            return i

for c in [20, 200]:
    best = 2 / (2 + c)
    print(c, round(best, 4), steps_needed(c, best))

*The lesson shows:*

```text
20 0.0909 24
200 0.0099 289
```

### The penguin valley

In [ ]:
flipper = np.array([181, 183, 185, 192, 193, 193,
                    197, 198, 205, 208, 216, 222], dtype=float)  # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g

def table(x):   # curvatures of the mean squared error for mass = w*x + b
    return 2 * np.array([[np.mean(x * x), np.mean(x)], [np.mean(x), 1.0]])

lo, hi = np.linalg.eigvalsh(table(flipper))
print(round(lo, 4), round(hi, 1), f"{hi / lo:.2e}")
print("limit", 2 / hi)

*The lesson shows:*

```text
0.0077 78515.2 1.02e+07
limit 2.5472788017166583e-05
```

The curvatures are 0.0077 and 78,515 (the larger sits a little above the 78,513 along $w$ alone, because the steepest direction is not exactly along $w$), a ratio of ten million, and the largest learning rate that does not blow up is $\frac{2}{78{,}515} = 0.0000255$. Run plain gradient descent at 99% of it, from $w = 0$, $b = 0$:

In [ ]:
def descend(x, lr, steps):
    w = b = 0.0
    for _ in range(steps):
        miss = w * x + b - mass
        w, b = w - lr * 2 * np.mean(miss * x), b - lr * 2 * np.mean(miss)
    return round(w, 3), round(b, 3), round(np.mean((w * x + b - mass) ** 2), 1)

for steps in [100, 100000]:
    print(steps, descend(flipper, 0.99 * 2 / hi, steps))

*The lesson shows:*

```text
100 (17.04, -0.027, 579618.0)
100000 (20.209, -111.914, 308170.2)
```

### Round the bowl by rescaling

The cure is to change the knobs’ meaning so that the bowl is round. Subtract the mean flipper length from every flipper, and divide by its **spread**, the typical distance from the mean (the standard deviation: average the twelve squared distances from the mean, then take the square root), 12.31 mm. The new input is $z$, the number of spreads above the average. Now the mean of $z$ is 0 and the mean of $z^2$ is 1, so the curvature along the slope is $2 \times 1 = 2$, along the shift it is 2, and the off-diagonal entry, $2 \times$ the mean of $z$, is 0. The table is $\begin{pmatrix} 2 & 0 \\ 0 & 2 \end{pmatrix}$: a round bowl. The limit is $\frac{2}{2} = 1$, and the rate $0.5$ makes the factor $1 - 0.5 \times 2 = 0$ in every direction: one step lands on the bottom.

In [ ]:
z = (flipper - flipper.mean()) / flipper.std()
print(np.round(table(z), 4).tolist())
print(descend(z, 0.5, 1))
print(round(48.9138 * flipper.std(), 2), round(mass.mean(), 1))

*The lesson shows:*

```text
[[2.0, 0.0], [0.0, 2.0]]
(602.098, 3862.5, 182842.1)
```

## 6.4 Bumpy ground: valleys, saddles and plateaus

A network’s loss is not a smooth bowl: it has valleys, saddles and flat patches. This lesson shows what a bowl guarantees, what a network does not, and what a two-dimensional picture hides.

[Read the lesson](https://aicourses.dev/mathematics/optimisation/bumpy-ground)

### What a network takes away

**Several valleys.** Take the one-knob loss $x^4 - 2x^2 + 0.5x$, which has two dips. Test the bowl rule on it: the line between $x = -1.2$ and $x = 1.0$ should lie above the curve if it were a bowl.

In [ ]:
import numpy as np

f = lambda x: x**4 - 2 * x**2 + 0.5 * x
slope = lambda x: 4 * x**3 - 4 * x + 0.5
for start in [-1.5, 1.5]:
    x = start
    for _ in range(200):
        x = x - 0.05 * slope(x)
    print(start, round(x, 4), round(f(x), 4))
print(round(f(-0.1), 4), round((f(-1.2) + f(1.0)) / 2, 4))

*The lesson shows:*

```text
-1.5 -1.0575 -1.5148
1.5 0.9304 -0.5167
-0.0699 -0.9532
```

**Twin copies.** A network’s hidden units are added together at the output, and adding does not care about order. So swap two hidden units, with their incoming weights, shifts and outgoing weights, and the network computes exactly the same function. If one setting is a bottom, its twin is a bottom at the same height. Here a two-unit network defines its own targets, so its loss is 0, and then its twin and the midpoint between them are scored:

In [ ]:
rng = np.random.default_rng(0)
net = lambda p, X: np.tanh(X @ p[0].T + p[1]) @ p[2].T + p[3]
p = [rng.normal(size=(2, 2)), rng.normal(size=2),
     rng.normal(size=(1, 2)), rng.normal(size=1)]   # W1, b1, W2, b2
X = rng.normal(size=(6, 2))
T = net(p, X)                                 # the targets it matches exactly
twin = [p[0][::-1], p[1][::-1], p[2][:, ::-1], p[3]]   # the units swapped
mid = [(a + b) / 2 for a, b in zip(p, twin)]
for q in [p, twin, mid]:
    print(round(np.mean((net(q, X) - T) ** 2), 4))

*The lesson shows:*

```text
0.0
0.0
0.0343
```

In [ ]:
x, y = 1.0, 0.001            # loss x**2 - y**2: slopes (2x, -2y)
for i in range(1, 31):
    x, y = x - 0.1 * 2 * x, y - 0.1 * (-2 * y)
    if i in (1, 10, 20, 30):
        print(i, round(x, 5), round(y, 5))
print([f"{1 - np.tanh(z) ** 2:.2e}" for z in [0, 2, 5, 10]])

*The lesson shows:*

```text
1 0.8 0.0012
10 0.10737 0.00619
20 0.01153 0.03834
30 0.00124 0.23738
['1.00e+00', '7.07e-02', '1.82e-04', '8.24e-09']
```

## 6.5 Momentum

Keeping a running average of past steps cancels the zig-zag and speeds up the crawl. It is the first fix for the troubles of the last two lessons.

[Read the lesson](https://aicourses.dev/mathematics/optimisation/momentum)

### By hand on the stretched bowl

Watch the two velocities. In $x$ the slope has the same sign every step, and the velocity builds: 2, 3.6, 4.68. In $y$ the first slope was +20 and the next −18, and the memory cancels most of it: the velocity falls from 18 to −1.8. The walker overshoots in $y$: the memory carries it past the floor first, and the opposing slopes then cancel the swing. The code repeats it:

In [ ]:
import numpy as np

x, y, vx, vy = 1.0, 1.0, 0.0, 0.0
for i in range(1, 4):
    vx, vy = 0.9 * vx + 2 * x, 0.9 * vy + 20 * y   # remember, then add
    x, y = x - 0.05 * vx, y - 0.05 * vy
    print(i, round(x, 4), round(y, 4))

*The lesson shows:*

```text
1 0.9 0.0
2 0.72 -0.9
3 0.486 -0.81
```

Does the memory pay off? On a much more stretched bowl, with curvatures 2 and 200, count the steps to bring the loss below 0.001. Plain descent at its best rate, $\frac{2}{2 + 200}$, is compared with momentum 0.9 at three rates:

In [ ]:
def steps_needed(lr, beta, c=200):
    x = y = 1.0
    vx = vy = 0.0
    for i in range(1, 100000):
        vx, vy = beta * vx + 2 * x, beta * vy + c * y
        x, y = x - lr * vx, y - lr * vy
        if not np.isfinite(x) or abs(y) > 1e6:
            return "flies apart"
        if x * x + c / 2 * y * y < 0.001:
            return i

for lr, beta in [(2 / 202, 0), (0.002, 0.9), (0.018, 0.9), (0.02, 0.9)]:
    print(round(lr, 4), beta, steps_needed(lr, beta))

*The lesson shows:*

```text
0.0099 0 289
0.002 0.9 64
0.018 0.9 95
0.02 0.9 flies apart
```

### Momentum on the digits

Back to the digit classifier of lesson 2. Plain descent at its best rate, 0.08, reached losses 0.574, 0.350 and 0.315 after 10, 100 and 500 steps. Momentum 0.9 at four rates:

In [ ]:
from sklearn.datasets import load_digits

np.seterr(over="ignore", invalid="ignore")
rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16
order = rng.permutation(len(X))
X, y = X[order], y[order]
A = np.hstack([X[:1297], np.ones((1297, 1))])
Y = np.eye(10)[y[:1297]]
loss = lambda P: ((A @ P - Y) ** 2).sum(axis=1).mean()
grad = lambda P: 2 / len(A) * A.T @ (A @ P - Y)
print(A.shape)

*The lesson shows:*

```text
(1297, 65)
```

In [ ]:
def train(lr, beta, steps=500):
    P, v = np.zeros((65, 10)), 0
    for i in range(1, steps + 1):
        v = beta * v + grad(P)
        P = P - lr * v
        if i in (10, 100, 500):
            print(f"  step {i}: {loss(P):.4g}", end="")
    print()

for lr in [0.05, 0.1, 0.16, 0.17]:
    print("rate", lr)
    train(lr, 0.9)

*The lesson shows:*

```text
rate 0.05
  step 10: 0.5046  step 100: 0.3128  step 500: 0.3082
rate 0.1
  step 10: 0.5214  step 100: 0.3096  step 500: 0.3078
rate 0.16
  step 10: 0.9181  step 100: 0.3087  step 500: 0.3075
rate 0.17
  step 10: 247.8  step 100: 8.33e+22  step 500: 1.406e+114
```

## 6.6 Mini-batches and noise

Real training uses a small random handful of examples at each step, which gives a cheap, noisy gradient. You see why that is good enough and what the noise costs.

[Read the lesson](https://aicourses.dev/mathematics/optimisation/mini-batches)

### How noisy is a handful?

Measure it on the digit classifier at its starting point, all knobs at zero, where the full gradient has length 2.308. For each batch size, draw 300 random batches and find how far each batch gradient is from the full one:

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

np.seterr(over="ignore", invalid="ignore")
rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16
order = rng.permutation(len(X))
X, y = X[order], y[order]
A = np.hstack([X[:1297], np.ones((1297, 1))])
Y = np.eye(10)[y[:1297]]
loss = lambda P: ((A @ P - Y) ** 2).sum(axis=1).mean()
grad = lambda P, A=A, Y=Y: 2 / len(A) * A.T @ (A @ P - Y)
print(np.linalg.norm(grad(np.zeros((65, 10)))).round(3))

*The lesson shows:*

```text
2.308
```

In [ ]:
P0 = np.zeros((65, 10))
full = grad(P0)
r2 = np.random.default_rng(1)          # a second generator for this demo
for B in [8, 32, 128, 512]:
    gaps = []
    for _ in range(300):
        idx = r2.choice(1297, B, replace=False)
        gaps.append(np.linalg.norm(grad(P0, A[idx], Y[idx]) - full))
    print(B, round(np.mean(gaps), 3))
batches = [grad(P0, A[i], Y[i]) for i in
           (r2.choice(1297, 32, replace=False) for _ in range(2000))]
print(round(np.linalg.norm(np.mean(batches, axis=0) - full), 3))

*The lesson shows:*

```text
8 2.722
32 1.316
128 0.639
512 0.26
0.031
```

### Training with handfuls

Now train with batches of 32 at four plain rates, 500 steps each. One pass over all 1,297 images is $\frac{1297}{32} = 40.5$ steps, called an **epoch**, so 500 steps are 12.3 epochs. A fresh random batch is drawn at each step, and the loss printed is measured on all 1,297 images:

In [ ]:
Xte = X[1297:]
Ate = np.hstack([Xte, np.ones((500, 1))])
acc = lambda P: ((Ate @ P).argmax(axis=1) == y[1297:]).mean()
def minibatch_run(lr, steps=500, size=32):
    P = np.zeros((65, 10))
    for i in range(1, steps + 1):
        idx = rng.choice(1297, size, replace=False)    # a random handful
        P = P - lr * grad(P, A[idx], Y[idx])
        if i in (10, 100, 500):
            print(f"  step {i}: {loss(P):.4g}", end="")
    print("  accuracy", acc(P))

for lr in [0.02, 0.05, 0.08, 0.3]:
    print("rate", lr)
    minibatch_run(lr)

*The lesson shows:*

```text
rate 0.02
  step 10: 0.7777  step 100: 0.4534  step 500: 0.3437  accuracy 0.942
rate 0.05
  step 10: 0.6619  step 100: 0.3907  step 500: 0.3353  accuracy 0.946
rate 0.08
  step 10: 0.6398  step 100: 0.4919  step 500: 0.3855  accuracy 0.91
rate 0.3
  step 10: 2.47e+14  step 100: 5.58e+152  step 500: nan  accuracy 0.11
```

What was bought? The same arithmetic as 500 batch steps would run only 12.3 full steps:

In [ ]:
P = np.zeros((65, 10))
for _ in range(12):
    P = P - 0.08 * grad(P)
print(round(loss(P), 4), round(500 * 32 / 1297, 1))

*The lesson shows:*

```text
0.5447
12.3
```

## 6.7 Adaptive steps and Adam

Adam divides each knob’s step by the recent size of its gradients, so every knob moves about one learning rate per step. This lesson works one step by hand and says what is and is not known about why it works so well.

[Read the lesson](https://aicourses.dev/mathematics/optimisation/adam)

### One step by hand

Now the three gradients of the Guess, and a third, 0.0005:

In [ ]:
import numpy as np

def first_step(g, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    m = (1 - b1) * g             # running average of g, started at 0
    v = (1 - b2) * g * g         # running average of g squared
    m_hat, v_hat = m / (1 - b1), v / (1 - b2)    # the correction at step 1
    return m, v, m_hat, v_hat, lr * m_hat / (np.sqrt(v_hat) + eps)

for g in [0.5, 500, 0.0005]:
    print(g, [float(f"{t:.6g}") for t in first_step(g)])

*The lesson shows:*

```text
0.5 [0.05, 0.00025, 0.5, 0.25, 0.001]
500 [50.0, 250.0, 500.0, 250000.0, 0.001]
0.0005 [5e-05, 2.5e-10, 0.0005, 2.5e-07, 0.00099998]
```

### Why the correction

Without the correction the first step would use $m_1 = 0.05$ and $v_1 = 0.00025$ directly. The memory has been started at 0, so $m_1$ is a tenth of the gradient and $v_1$ a thousandth of its square, whose root is about a thirtieth. A tenth divided by a thirtieth is 3.16. The code checks it, and does step 2 with a slope of $-0.5$ (opposite) and of $+0.5$ (the same):

In [ ]:
m, v = 0.05, 0.00025                 # after step 1, gradient 0.5
for g in [-0.5, 0.5]:                # next gradient: opposite, or same
    m2, v2 = 0.9 * m + 0.1 * g, 0.999 * v + 0.001 * g * g
    m_hat, v_hat = m2 / (1 - 0.9**2), v2 / (1 - 0.999**2)
    step = 0.001 * m_hat / (np.sqrt(v_hat) + 1e-8)
    print(g, round(m_hat, 6), round(v_hat, 4), f"{step:.4g}")
nobc = [0.001 * (0.1 * g) / np.sqrt(0.001 * g * g) for g in [0.5, 500, 0.0005]]
print(np.round(nobc, 5))

*The lesson shows:*

```text
-0.5 -0.026316 0.25 -5.263e-05
0.5 0.5 0.25 0.001
[0.00316 0.00316 0.00316]
```

### Adam on the digits

The digit classifier again, with Adam at three rates, the paper’s default 0.001 first:

In [ ]:
from sklearn.datasets import load_digits

np.seterr(over="ignore", invalid="ignore")
rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16
order = rng.permutation(len(X))
X, y = X[order], y[order]
A = np.hstack([X[:1297], np.ones((1297, 1))])
Y = np.eye(10)[y[:1297]]
Ate = np.hstack([X[1297:], np.ones((500, 1))])
loss = lambda P: ((A @ P - Y) ** 2).sum(axis=1).mean()
grad = lambda P: 2 / len(A) * A.T @ (A @ P - Y)
acc = lambda P: ((Ate @ P).argmax(axis=1) == y[1297:]).mean()
print(A.shape)

*The lesson shows:*

```text
(1297, 65)
```

In [ ]:
def adam(lr, steps=500, b1=0.9, b2=0.999, eps=1e-8):
    P, m, v = np.zeros((65, 10)), 0, 0
    for i in range(1, steps + 1):
        g = grad(P)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        m_hat, v_hat = m / (1 - b1 ** i), v / (1 - b2 ** i)
        P = P - lr * m_hat / (np.sqrt(v_hat) + eps)
        if i in (10, 100, 500):
            print(f"  step {i}: {loss(P):.4g}", end="")
    print("  accuracy", acc(P))

for lr in [0.001, 0.01, 0.1]:
    print("rate", lr)
    adam(lr)

*The lesson shows:*

```text
rate 0.001
  step 10: 0.8305  step 100: 0.4264  step 500: 0.3146  accuracy 0.946
rate 0.01
  step 10: 0.5344  step 100: 0.3123  step 500: 0.3049  accuracy 0.942
rate 0.1
  step 10: 9.823  step 100: 0.3157  step 500: 0.3048  accuracy 0.94
```

### Weight decay the AdamW way

$\eta_t$ is a schedule multiplier (lesson 8), taken as 1 here, $\alpha$ is Adam’s rate and $\lambda$ is the decay. A hand check with $w = 2$ and $\lambda = 0.001$: the decay adds $0.001 \times 2 = 0.002$ to the gradient in the L2 version, where the penalty is simply added to the loss. At step 1 Adam divides the whole gradient, decay included, by its own size, so the decay’s part of the step is the rate times $\lambda w$ divided by that gradient (the code calls the weight `theta`). For two weights with first gradients 0.5 and 0.0005:

In [ ]:
theta, lam, lr = 2.0, 0.001, 0.001
for g in [0.5, 0.0005]:
    g_l2 = g + lam * theta               # L2: the decay joins the gradient
    print(g, f"{lr * lam * theta / g_l2:.2e}")   # decay’s share

*The lesson shows:*

```text
0.5 3.98e-06
0.0005 8.00e-04
```

## 6.8 Schedules, warmup and clipping

Large training runs start gently, decay later and cap the length of the gradient. These three habits, warmup, schedules and clipping, are what keep a big run from blowing up.

[Read the lesson](https://aicourses.dev/mathematics/optimisation/schedules-warmup-clipping)

### Warmup

**Warmup** starts the learning rate near zero and raises it to its peak over the first steps. The Transformer paper (Vaswani and colleagues, 2017) gave a formula you built in lesson 01.10: the rate at step $s$ is $d_{\text{model}}^{-0.5} \cdot \min(s^{-0.5},\ s \cdot \text{warmup}^{-1.5})$, with $d_{\text{model}} = 512$ and a warmup of 4,000 steps. Below step 4,000 the second term is the smaller, and the rate grows in step with $s$. Above it, the rate falls as $s^{-0.5}$.

In [ ]:
import numpy as np

d_model, warmup = 512, 4000
lr = lambda s: d_model ** -0.5 * min(s ** -0.5, s * warmup ** -1.5)
print([f"{lr(s):.3g}" for s in [1, 2000, 4000, 16000]])

cosine = lambda t, T, top: top * 0.5 * (1 + np.cos(np.pi * t / T))
print([round(cosine(t, 500, 0.08), 4) for t in [0, 125, 250, 375, 500]])

*The lesson shows:*

```text
['1.75e-07', '0.000349', '0.000699', '0.000349']
[0.08, 0.0683, 0.04, 0.0117, 0.0]
```

Why start gently? In lesson 7 you saw that Adam’s first step moves every knob a full rate, whatever its gradient. At a rate of 0.1 that is a big first move for all 650 knobs at once. Try it on the digit classifier, first without warmup, then with a 10-step warmup, then with the cosine decay as well:

In [ ]:
from sklearn.datasets import load_digits

np.seterr(over="ignore", invalid="ignore")
rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16
order = rng.permutation(len(X))
X, y = X[order], y[order]
A = np.hstack([X[:1297], np.ones((1297, 1))])
Y = np.eye(10)[y[:1297]]
Ate = np.hstack([X[1297:], np.ones((500, 1))])
loss = lambda P: ((A @ P - Y) ** 2).sum(axis=1).mean()
grad = lambda P, A=A, Y=Y: 2 / len(A) * A.T @ (A @ P - Y)
acc = lambda P: ((Ate @ P).argmax(axis=1) == y[1297:]).mean()
print(A.shape)

*The lesson shows:*

```text
(1297, 65)
```

In [ ]:
def adam(lr_at, steps=500):
    P, m, v, h = np.zeros((65, 10)), 0, 0, []
    for i in range(1, steps + 1):
        g = grad(P)
        m, v = 0.9 * m + 0.1 * g, 0.999 * v + 0.001 * g ** 2
        m_hat, v_hat = m / (1 - 0.9 ** i), v / (1 - 0.999 ** i)
        P = P - lr_at(i) * m_hat / (np.sqrt(v_hat) + 1e-8)
        h.append(loss(P))
    return h
runs = {"constant 0.1": lambda i: 0.1,
        "warmup 10 steps": lambda i: 0.1 * min(1, i / 10),
        "warmup + cosine": lambda i: 0.1 * min(1, i / 10) * cosine(i, 500, 1)}
for name, f in runs.items():
    h = adam(f)
    print(name, [f"{h[i - 1]:.4g}" for i in (10, 100, 500)])

*The lesson shows:*

```text
constant 0.1 ['9.823', '0.3157', '0.3048']
warmup 10 steps ['0.506', '0.3274', '0.3311']
warmup + cosine ['0.5066', '0.3308', '0.3047']
```

### Decay

Decay matters most with mini-batches, whose steps carry noise that the rate multiplies. Train with batches of 32 at a constant rate of 0.08, and then with the same rate decayed by a cosine over 500 steps, using the same batches in both runs, from three random seeds:

In [ ]:
def minibatch(lr_at, seed, steps=500):
    r = np.random.default_rng(seed)       # same batches for every schedule
    P = np.zeros((65, 10))
    for i in range(steps):
        idx = r.choice(1297, 32, replace=False)
        P = P - lr_at(i) * grad(P, A[idx], Y[idx])
    return round(float(loss(P)), 4)

for seed in [1, 2, 3]:
    print(seed, minibatch(lambda i: 0.08, seed),
          minibatch(lambda i: cosine(i, 500, 0.08), seed))

*The lesson shows:*

```text
1 0.621 0.3251
2 0.5369 0.325
3 0.3989 0.3257
```

### Clipping

Earlier in this module, plain descent at rate 0.5 diverged on the digit classifier. Now clip the gradient, at limits 1 and 0.5, and run the same rate:

In [ ]:
def clip(g, limit):
    size = np.linalg.norm(g)
    return g * min(1, limit / size)

print(clip(np.array([3.0, 4.0]), 1.0))
for limit in [None, 1.0, 0.5]:
    P = np.zeros((65, 10))
    for i in range(1, 501):
        g = grad(P)
        P = P - 0.5 * (g if limit is None else clip(g, limit))
        if i in (10, 100, 500):
            print(f"  step {i}: {loss(P):.4g}", end="")
    print("  limit", limit, "accuracy", acc(P))

*The lesson shows:*

```text
[0.6 0.8]
  step 10: 2.387e+19  step 100: 6.402e+202  step 500: nan  limit None accuracy 0.11
  step 10: 0.7834  step 100: 0.6582  step 500: 0.6368  limit 1.0 accuracy 0.95
  step 10: 0.7077  step 100: 0.5135  step 500: 0.482  limit 0.5 accuracy 0.95
```

## 6.9 A glance at curvature-aware methods

Newton’s step lands on a parabola’s bottom in one go, but the table of curvatures it needs is far too big for a model. This optional glance shows what a curvature-aware method would give and why it is not used.

[Read the lesson](https://aicourses.dev/mathematics/optimisation/curvature-aware-methods)

### The Newton step

That is gradient descent with the learning rate set to one over the curvature, so the step is the slope divided by how fast the slope changes. A parabola’s slope is a straight line, and this step follows it down to where it crosses zero, which is the bottom. For $L = w^2$ at $w = 5$: slope 10, curvature 2, step to $5 - \frac{10}{2} = 0$. On the penguin bowl of lesson 04.8 the slope at $w = 0$ is $-1.4823$ and the curvature is 3.0304, so the step lands at $0 + \frac{1.4823}{3.0304} = 0.4891$, the best $w$, in one move. The code checks both, and tries the hilltop $-x^2$ too:

In [ ]:
import numpy as np

x = 5.0
print(x - (2 * x) / 2)                  # bowl x**2: slope 2x, curvature 2
print(round(0 - (-1.4823) / 3.0304, 4)) # penguins, from w = 0
print(x - (-2 * x) / (-2))              # hilltop -x**2: slope -2x, curv. -2

*The lesson shows:*

```text
0.0
0.4891
0.0
```

### It works on the digits

For the digit classifier the table is $\mathbf{H} = \frac{2}{1297}\mathbf{A}^\top \mathbf{A}$, with 65 rows and columns: each digit’s 65 knobs form a separate bowl, and all ten share this table. Module 10 explains where it comes from. Three pixels are never inked in the training digits, so the table has no true inverse; `np.linalg.pinv` stands in: a table goes in, and a stand-in for its inverse comes out, ignoring those three directions. One step from zeros:

In [ ]:
from sklearn.datasets import load_digits
rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16
order = rng.permutation(len(X))
X, y = X[order], y[order]
A = np.hstack([X[:1297], np.ones((1297, 1))])
Y = np.eye(10)[y[:1297]]
Ate = np.hstack([X[1297:], np.ones((500, 1))])
loss = lambda P: ((A @ P - Y) ** 2).sum(axis=1).mean()
grad = lambda P: 2 / len(A) * A.T @ (A @ P - Y)
acc = lambda P: ((Ate @ P).argmax(axis=1) == y[1297:]).mean()
H = 2 / len(A) * A.T @ A                 # the table of curvatures
P1 = np.zeros((65, 10)) - np.linalg.pinv(H) @ grad(np.zeros((65, 10)))
print(H.shape, round(loss(P1), 5), acc(P1))

*The lesson shows:*

```text
(65, 65) 0.30474 0.944
```

### The table is too big

For $n$ knobs the table has $n \times n$ entries. The digit classifier has 650 knobs, a table of $650^2 = 422{,}500$ entries if you do not notice that it splits into ten. Then one has to invert the table, which takes work that grows with the cube of $n$. Count the table for a large model:

In [ ]:
print(650 ** 2, f"{405e9 ** 2:.3g}")

*The lesson shows:*

```text
422500 1.64e+23
```

## 6.10 Project: train the digit classifier

You train Module 03’s matrix from zeros three ways, with plain descent, momentum and Adam, and compare their curves. It is the first time your own classifier learns its weights.

[Read the lesson](https://aicourses.dev/mathematics/optimisation/project-train-the-classifier)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Step 1: the model and the loss

The classifier is the matrix and shift of Module 03, written as one table $\mathbf{P}$ of 65 rows and 10 columns by adding a 1 to every image, with the loss and gradient of lesson 2. The first line of output also scores the ten templates: 0.9, the number to beat.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

np.seterr(over="ignore", invalid="ignore")
rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16
order = rng.permutation(len(X))
X, y = X[order], y[order]
Xtr, ytr, Xte, yte = X[:1297], y[:1297], X[1297:], y[1297:]
A = np.hstack([Xtr, np.ones((1297, 1))])      # images and a 1 for the shift
Ate = np.hstack([Xte, np.ones((500, 1))])
Y = np.eye(10)[ytr]
loss = lambda P: ((A @ P - Y) ** 2).sum(axis=1).mean()
grad = lambda P: 2 / len(A) * A.T @ (A @ P - Y)
acc = lambda P: ((Ate @ P).argmax(axis=1) == yte).mean()

W = np.array([Xtr[ytr == k].mean(axis=0) for k in range(10)])   # Module 03
print(A.shape, round(((Xte @ W.T).argmax(axis=1) == yte).mean(), 3))

*The lesson shows:*

```text
(1297, 65) 0.9
```

### Step 2: three ways to walk

Each method is a function that takes the knobs, a place to keep its memory and the step number, and returns the new knobs. `train` runs one for 500 steps from zeros and keeps the loss after every step.

In [ ]:
def train(step, steps=500):
    P, state, history = np.zeros((65, 10)), {}, []
    for i in range(1, steps + 1):
        P = step(P, state, i)
        history.append(loss(P))
    return P, np.array(history)

def plain(lr):
    return lambda P, s, i: P - lr * grad(P)

def momentum(lr, beta=0.9):
    def step(P, s, i):
        s["v"] = beta * s.get("v", 0) + grad(P)
        return P - lr * s["v"]
    return step

def adam(lr, b1=0.9, b2=0.999, eps=1e-8):
    def step(P, s, i):
        g = grad(P)
        s["m"] = b1 * s.get("m", 0) + (1 - b1) * g
        s["v"] = b2 * s.get("v", 0) + (1 - b2) * g ** 2
        m_hat, v_hat = s["m"] / (1 - b1 ** i), s["v"] / (1 - b2 ** i)
        return P - lr * m_hat / (np.sqrt(v_hat) + eps)
    return step
print(len(train(plain(0.08))[1]))

*The lesson shows:*

```text
500
```

### Step 3: run them

Plain descent at 0.08 (lesson 2’s best), momentum 0.9 at 0.05 (lesson 5), and Adam at 0.01, with Adam at the paper’s default 0.001 as a fourth. For each, the loss after 10, 100 and 500 steps and the accuracy on the 500 held-out digits.

In [ ]:
runs = {"plain 0.08": plain(0.08), "momentum 0.05": momentum(0.05),
        "Adam 0.01": adam(0.01), "Adam 0.001": adam(0.001)}
curves = {}
for name, step in runs.items():
    P, curves[name] = train(step)
    h = curves[name]
    print(f"{name:14s} {h[9]:.4f} {h[99]:.4f} {h[499]:.4f}  acc {acc(P)}")
print({n: round(float(h.min()), 4) for n, h in curves.items()})

*The lesson shows:*

```text
plain 0.08     0.5736 0.3500 0.3147  acc 0.952
momentum 0.05  0.5046 0.3128 0.3082  acc 0.946
Adam 0.01      0.5344 0.3123 0.3049  acc 0.942
Adam 0.001     0.8305 0.4264 0.3146  acc 0.946
{'plain 0.08': 0.3147, 'momentum 0.05': 0.3082, 'Adam 0.01': 0.3049, 'Adam 0.001': 0.3146}
```

Now the plot. matplotlib is used here and nowhere else in the module, because this is the one place you must see your own curves. A log scale on the loss shows both the early gap and the late one.

In [ ]:
import matplotlib.pyplot as plt

for name, h in curves.items():
    plt.plot(range(1, 501), h, label=name)
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("training loss")
plt.legend()
plt.show()

### Find a rate that diverges

You have already seen 0.5 diverge, in lesson 2, and the limit $\frac{2}{22.9} = 0.0873$. Run plain descent at three rates around it:

In [ ]:
for lr in [0.0865, 0.0917, 0.5]:
    P, h = train(plain(lr))
    print(lr, f"{h[9]:.4g} {h[99]:.4g} {h[499]:.4g}", acc(P))

*The lesson shows:*

```text
0.0865 0.6259 0.3489 0.314 0.952
0.0917 1.213 1.761e+07 1.744e+40 0.116
0.5 2.387e+19 6.402e+202 nan 0.11
```